## Notebook - Inital Data Exploration (EDA)

In [ ]:
# Locate the project whether the notebook starts in the root or scripts folder.
from pathlib import Path
project_root = Path.cwd()
if not (project_root / "data" / "processed").is_dir():
    project_root = project_root.parent


In [ ]:
import pandas as pd 
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path

In [ ]:
data = pd.read_csv(project_root / "data/processed/yearly_cases_2021_2025.csv")
data.head()


___

**Analysis #1**

Look at the yearly trend of disease categories exclusing COVID-19

In [ ]:
first_plot_data = data.groupby(by = ["Disease Category", "Year"], as_index = False )[["Yearly Cases", "January", "February", "March", 
"April", "May", "June", "July", "August", "September", "October", "November", "December"]].sum()
first_plot_data.head(10)

In [ ]:
for cat in first_plot_data["Disease Category"].unique():
    cat_data = first_plot_data[first_plot_data["Disease Category"] == cat]
    
    plt.figure()
    plt.plot(
        cat_data["Year"],
        cat_data["Yearly Cases"], 
        marker = 'o'
    )
    plt.title(cat)
    plt.xlabel("Year")
    plt.ylabel("Yearly Cases")
    plt.show()

**Would be interesting to see monthly trends**

In [ ]:
months = ["January", "February", "March", "April","May", "June", "July", "August", "September", "October", "November", "December"]

monthly_data = first_plot_data.melt(
    id_vars=["Disease Category", "Year"], value_vars=months, var_name="Month", value_name="Cases")

monthly_data["Date"] = pd.to_datetime(
    monthly_data["Year"].astype(str) + " " + monthly_data["Month"], format="%Y %B"
)
monthly_data

In [ ]:
for cat in monthly_data["Disease Category"].unique():
    cat_data = monthly_data[monthly_data["Disease Category"] == cat].sort_values("Date")
    
    plt.plot(
        cat_data["Date"],
        cat_data["Cases"],
        marker="o"
    )

    plt.title(cat)
    plt.xlabel("Date")
    plt.ylabel("Cases")
    plt.xticks(rotation=45)

    plt.show()

___ 

## Vaccine Preventable Diseases

In [ ]:
vacc_data = data.loc[data["Disease Category"] == "Vaccine Preventable Diseases"].copy()
# Keep published rates such as <0.1 unchanged; the plots below use case counts.
vacc_data.head(10)


### Monthly disease contributions

The individual-disease dates must come from `vacc_mon_data` itself. Assigning
`monthly_data` dates aligns unrelated row indices and leaves many dates missing.
The checks below verify that every disease-month has the correct date and that
individual counts reproduce the earlier category trend for all 60 months.
Annual totals are not used here: revised 2025 totals can differ from monthly sums.


In [ ]:
# Monthly comparisons use counts, not annual rates.


In [ ]:
vacc_mon_data = vacc_data.melt(
    id_vars=["Disease", "Year"], value_vars=months,
    var_name="Month", value_name="Cases"
)
# Build dates from THIS table, with an explicit format to avoid ambiguous parsing.
vacc_mon_data["Date"] = pd.to_datetime(
    vacc_mon_data["Year"].astype(str) + " " + vacc_mon_data["Month"],
    format="%Y %B"
)
assert vacc_mon_data["Date"].notna().all()
assert not vacc_mon_data.duplicated(["Disease", "Date"]).any()
vacc_mon_data = vacc_mon_data.sort_values(["Disease", "Date"])
vacc_mon_data.head()


In [ ]:
# Reconcile individual diseases against the category totals plotted above.
disease_totals = vacc_mon_data.groupby("Date")["Cases"].sum(min_count=1)
category_totals = monthly_data.loc[
    monthly_data["Disease Category"] == "Vaccine Preventable Diseases"
].set_index("Date")["Cases"].sort_index()
reconciliation = pd.concat(
    [category_totals.rename("Category total"), disease_totals.rename("Disease sum")],
    axis=1
)
reconciliation["Difference"] = reconciliation["Disease sum"] - reconciliation["Category total"]
assert len(reconciliation) == 60
assert reconciliation.notna().all().all()
assert reconciliation["Difference"].eq(0).all()
print("All 60 monthly category totals equal the sum of the individual disease counts.")

# One column per disease makes each peak's composition easy to inspect.
contributions = vacc_mon_data.pivot(index="Date", columns="Disease", values="Cases")
peak_dates = category_totals.nlargest(10).index
peak_breakdown = contributions.loc[peak_dates].copy()
peak_breakdown.insert(0, "Category total", category_totals.loc[peak_dates])
peak_breakdown.insert(1, "Influenza share (%)",
                      (100 * peak_breakdown["Influenza"] / peak_breakdown["Category total"]).round(1))
peak_breakdown


In [ ]:
# Separate figures reveal each disease's timing on its own vertical scale.
# Compare absolute contributions using the stacked plot below.
for disease, disease_data in vacc_mon_data.groupby("Disease", sort=True):
    disease_data = disease_data.sort_values("Date")
    fig, ax = plt.subplots(figsize=(10, 3))
    ax.plot(disease_data["Date"], disease_data["Cases"], marker="o", markersize=3)
    ax.set(title=disease, xlabel="Date", ylabel="Monthly cases")
    ax.set_ylim(bottom=0)
    fig.autofmt_xdate()
    fig.tight_layout()
    plt.show()
    plt.close(fig)


### Which diseases make up the spikes?

The stacked areas add up to the category total (black line). The second panel
removes influenza to reveal smaller changes in the remaining diseases. Zero
contributions for diseases absent from a year's report mean no contribution to
the available-data total, not confirmed absence of disease. Missing counts in
reported disease rows are checked before plotting rather than replaced with zero.


In [ ]:
assert vacc_mon_data["Cases"].notna().all(), "Investigate missing monthly counts before stacking."
# NaNs introduced by pivoting are absent disease-year records, not source counts.
plot_contributions = contributions.fillna(0)
order = plot_contributions.sum().sort_values(ascending=False).index
plot_contributions = plot_contributions[order]
fig, axes = plt.subplots(2, 1, figsize=(13, 9), sharex=True)
colors = plt.get_cmap("tab20").colors
axes[0].stackplot(plot_contributions.index, plot_contributions.to_numpy().T,
                  labels=plot_contributions.columns, colors=colors)
axes[0].plot(category_totals.index, category_totals, color="black", linewidth=1.3,
             label="Category total")
axes[0].set(title="Vaccine preventable diseases: monthly contributions", ylabel="Monthly cases")
axes[0].legend(loc="upper left", bbox_to_anchor=(1.01, 1), fontsize=8)
other_diseases = plot_contributions.drop(columns="Influenza")
other_colors = [colors[i % len(colors)] for i, name in enumerate(plot_contributions.columns)
                if name != "Influenza"]
axes[1].stackplot(other_diseases.index, other_diseases.to_numpy().T, colors=other_colors)
axes[1].set(title="Same months, excluding influenza", ylabel="Monthly cases", xlabel="Date")
fig.tight_layout()
plt.show()
plt.close(fig)
